In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import numpy as np
from Clean_Data import get_cleaned_vanguard_data
from scipy.stats import chi2_contingency

In [ ]:
df = get_cleaned_vanguard_data(True)
df

Analyzing chronological journeys...


,client_id,path_category,variation,age,gender,tenure_years,account_balance
1,555,perfect_path,Test,29.5,Unknown,3.0,25454.66
2,647,perfect_path,Test,57.5,M,12.0,30525.80
4,934,dropped_in_between,Test,51.0,F,9.0,32522.88
5,1028,confused,Control,36.0,M,12.0,103520.22
6,1104,dropped_in_between,Control,48.0,Unknown,5.0,154643.94
...,...,...,...,...,...,...,...
70013,9999150,confused,Test,30.0,Unknown,5.0,97141.71
70015,9999400,perfect_path,Test,28.5,Unknown,7.0,51787.04
70016,9999626,dropped_in_between,Test,35.0,M,9.0,36642.88
70017,9999729,confused,Test,31.0,F,10.0,107059.74


In [7]:
def check_succesful_path(x):
    list=['perfect_path','successful_with_repeats']
    return x in list

client_level = df.groupby('client_id').agg({
    'variation': 'first',
    'age': 'first',
    'path_category': lambda x: check_succesful_path(x.iloc[0])
}).rename(columns={'path_category': 'is_completed'}).reset_index()

client_level = client_level.dropna(subset=['variation', 'age'])

bins = [0, 25, 40, 60, 120]
labels = ['Gen Z (Under 25)', 'Millennials (26-40)', 'Gen X (41-60)', 'Boomers/Retirees (60+)']
client_level['age_group'] = pd.cut(client_level['age'], bins=bins, labels=labels)

client_level


,client_id,variation,age,is_completed,age_group
0,555,Test,29.5,True,Millennials (26-40)
1,647,Test,57.5,True,Gen X (41-60)
2,934,Test,51.0,False,Gen X (41-60)
3,1028,Control,36.0,False,Millennials (26-40)
4,1104,Control,48.0,False,Gen X (41-60)
...,...,...,...,...,...
50151,9999150,Test,30.0,False,Millennials (26-40)
50152,9999400,Test,28.5,True,Millennials (26-40)
50153,9999626,Test,35.0,False,Millennials (26-40)
50154,9999729,Test,31.0,False,Millennials (26-40)


In [8]:
client_level['age_group'].cat.categories

Index(['Gen Z (Under 25)', 'Millennials (26-40)', 'Gen X (41-60)',
       'Boomers/Retirees (60+)'],
      dtype='str')

In [9]:
for age_cat in client_level['age_group'].cat.categories:

    print(f"For Age Category: {age_cat}")
    group_data = client_level[client_level['age_group'] == age_cat]
    variation_test_data = group_data[group_data['variation'] == 'Test']
    variation_test_success = variation_test_data['is_completed'].sum()
    variation_test_fail = len(variation_test_data) - variation_test_success
    variation_control_data = group_data[group_data['variation'] == 'Control']
    variation_control_success = variation_control_data['is_completed'].sum()
    variation_control_fail = len(variation_control_data) - variation_control_success
    
    if len(variation_test_data) == 0 or len(variation_control_data) == 0:
        print("  Not enough data for this category.\n")
        continue
    
    observed = np.array([
        [variation_test_success, variation_test_fail],
        [variation_control_success, variation_control_fail]
    ])
    
    chi2_stat, p_val, dof, expected = chi2_contingency(observed)

    test_rate = variation_test_success / len(variation_test_data)
    control_rate = variation_control_success / len(variation_control_data)
    
    print(f"  Test Completion Rate:    {test_rate:.2%}")
    print(f"  Control Completion Rate: {control_rate:.2%}")
    print(f"  P-value: {p_val:.4e}")
    
    if p_val < 0.05:
        if test_rate > control_rate:
            print(f"Conclusion: The Test UI performed SIGNIFICANTLY BETTER for {age_cat}.\n")
        else:
            print(f"Conclusion: The Test UI performed SIGNIFICANTLY WORSE for {age_cat}.\n")
    else:
        print(f"➖ Conclusion: NO significant difference between Test and Control for {age_cat}.\n")


For Age Category: Gen Z (Under 25)
  Test Completion Rate:    48.28%
  Control Completion Rate: 40.12%
  P-value: 1.2846e-06
Conclusion: The Test UI performed SIGNIFICANTLY BETTER for Gen Z (Under 25).

For Age Category: Millennials (26-40)
  Test Completion Rate:    50.16%
  Control Completion Rate: 44.55%
  P-value: 1.2555e-11
Conclusion: The Test UI performed SIGNIFICANTLY BETTER for Millennials (26-40).

For Age Category: Gen X (41-60)
  Test Completion Rate:    44.45%
  Control Completion Rate: 42.72%
  P-value: 1.4612e-02
Conclusion: The Test UI performed SIGNIFICANTLY BETTER for Gen X (41-60).

For Age Category: Boomers/Retirees (60+)
  Test Completion Rate:    39.05%
  Control Completion Rate: 35.61%
  P-value: 1.0902e-04
Conclusion: The Test UI performed SIGNIFICANTLY BETTER for Boomers/Retirees (60+).



In [10]:
test_group = client_level[client_level['variation'] == 'Test']

age_completed = test_group[test_group['is_completed'] == True]['age']
age_failed = test_group[test_group['is_completed'] == False]['age']

t_stat, p_val = stats.ttest_ind(age_completed, age_failed)

print(f"Average Age (Completed): {age_completed.mean():.2f}")
print(f"Average Age (Failed): {age_failed.mean():.2f}")
print("-" * 30)
print(f"T-statistic: {t_stat:.4f}")
print(f"P-value: {p_val:.4e}")
print("-" * 30)

if p_val < 0.05:
    print("Conclusion: Reject H0.")
    print("There is a statistically significant difference in age between those who complete the new UI and those who drop off.")
else:
    print("Conclusion: Fail to reject H0.")
    print("There is NO significant difference in age between those who complete the new UI and those who drop off.")


Average Age (Completed): 45.59
Average Age (Failed): 48.75
------------------------------
T-statistic: -16.7482
P-value: 1.2163e-62
------------------------------
Conclusion: Reject H0.
There is a statistically significant difference in age between those who complete the new UI and those who drop off.


The new Test UI is highly successful and should be rolled out to all users. It significantly improves completion rates for every age group, including our senior demographics. 
However, as older users are still slightly more likely to drop off than younger users, future iterations of the UI could focus on accessibility or simplified flows to further assist the 50+ demographic."